agent

In [2]:
from google.adk.agents import Agent
from google.adk.apps import App
from google.adk.models import Gemini


def get_weather(city: str) -> str:
    """A simple, simulated tool to get the weather for a city.

    Args:
        city: The city to get the weather for, e.g., "San Francisco".

    Returns:
        A string with the simulated weather for the city.
    """
    if "san francisco" in city.lower():
        return "It's 65 degrees and foggy in San Francisco."
    return f"It's 72 degrees and sunny in {city}."


root_agent = Agent(
    name="root_agent",
    model=Gemini(model="gemini-1.0-pro"),
    instruction="You are a helpful weather assistant. Use your tools to provide weather information.",
    tools=[get_weather],
)

app = App(
    root_agent=root_agent,
    name="app",
)


agent engine app

In [ ]:
import logging
import os
from typing import Any

import vertexai
from dotenv import load_dotenv
from google.adk.artifacts import GcsArtifactService, InMemoryArtifactService
from google.cloud import logging as google_cloud_logging
from vertexai.agent_engines.templates.adk import AdkApp

from app.agent import app as adk_app
from app.app_utils.telemetry import setup_telemetry
from app.app_utils.typing import Feedback

# Load environment variables from .env file at runtime
load_dotenv()


class AgentEngineApp(AdkApp):
    def set_up(self) -> None:
        """Initialize the agent engine app with logging and telemetry."""
        setup_telemetry()
        super().set_up()
        logging.basicConfig(level=logging.INFO)
        logging_client = google_cloud_logging.Client()
        self.logger = logging_client.logger(__name__)
        if gemini_location:
            os.environ["GOOGLE_CLOUD_LOCATION"] = gemini_location

    def register_feedback(self, feedback: dict[str, Any]) -> None:
        """Collect and log feedback."""
        feedback_obj = Feedback.model_validate(feedback)
        self.logger.log_struct(feedback_obj.model_dump(), severity="INFO")

    def register_operations(self) -> dict[str, list[str]]:
        """Registers the operations of the Agent."""
        operations = super().register_operations()
        operations[""] = operations.get("", []) + ["register_feedback"]
        return operations


gemini_location = os.environ.get("GOOGLE_CLOUD_LOCATION")
logs_bucket_name = os.environ.get("LOGS_BUCKET_NAME")
agent_engine = AgentEngineApp(
    app=adk_app,
    artifact_service_builder=lambda: (
        GcsArtifactService(bucket_name=logs_bucket_name)
        if logs_bucket_name
        else InMemoryArtifactService()
    ),
)


requirements

In [ ]:
google-cloud-aiplatform
google-adk
google-cloud-logging
python-dotenv
requests
google-auth

In [ ]:
Deplployment

In [ ]:
import google.auth
import vertexai
from vertexai import agent_engines
from vertexai.agent_engines.templates.adk import AdkApp
from google.adk.agents import Agent
from google.adk.models import Gemini


# --- Agent Definition ---
# By defining the agent and its tools directly in this script, we create a
# self-contained and easy-to-manage deployment solution.

def get_weather(city: str) -> str:
    """A simple, simulated tool to get the weather for a city.

    Args:
        city: The city to get the weather for, e.g., "San Francisco".

    Returns:
        A string with the simulated weather for the city.
    """
    if "san francisco" in city.lower():
        return "It's 65 degrees and foggy in San Francisco."
    return f"It's 72 degrees and sunny in {city}."


root_agent = Agent(
    name="root_agent",
    model=Gemini(model="gemini-1.0-pro"),
    instruction="You are a helpful weather assistant. Use your tools to provide weather information.",
    tools=[get_weather],
)


def deploy_weather_agent():
    """Defines and deploys a simple weather agent to Vertex AI Agent Engine."""
    project_id = "qwiklabs-gcp-01-0c244accce49"
    location = "us-central1"
    staging_bucket = f"gs://{project_id}"

    # Initialize Vertex AI with the project, location, and staging bucket.
    # This is the correct and necessary step for configuration.
    vertexai.init(
        project=project_id,
        location=location,
        staging_bucket=staging_bucket,
    )

    print(f"Deploying agent to project '{project_id}' in '{location}'...")

    # Wrap the agent in an AdkApp for deployment.
    deployment = AdkApp(agent=root_agent, enable_tracing=True)

    # Define the requirements for the agent.
    requirements = [
        # Explicitly list dependencies for clarity and robustness.
        "google-cloud-aiplatform[adk,agent_engines]",
    ]

    try:
        # Deploy the agent using agent_engines.create()
        remote_app = agent_engines.create(
            deployment,
            requirements=requirements,
            display_name="Simple Weather Agent",
        )

        print("\nAgent deployed successfully!")
        print(f"  Resource Name: {remote_app.resource_name}")
        print(f"  Display Name: {remote_app.display_name}")

    except Exception as e:
        print(f"\nAn error occurred during deployment: {e}")


if __name__ == "__main__":
    deploy_weather_agent()